# 1. What is Machine Learning?

By the end, you will explain a dataset, a prediction, a learned parameter, training, and testing. You will build a tiny prediction rule by hand and measure its mistakes. No mathematics or Python experience is required. Read the prose first; the code is an inspectable calculator, and later lessons teach its syntax carefully. Allow two hours, including exercises.

## 2. What problem does this solve?

A bakery must decide how many loaves to prepare tomorrow. Too few disappoint customers; too many create waste. A handwritten rule might always prepare ten. Machine learning offers another approach: use recorded examples to choose parts of a prediction rule. It is useful when patterns are easier to demonstrate with examples than to specify completely as instructions.

Learning does not mean a computer understands bread. It means a procedure changes numerical settings using data. The choice of data, objective, and allowed rules comes from people. A computer can learn a very consistent pattern that is irrelevant or harmful if those choices are wrong.

## 3. Intuition and visualization

Imagine guessing a friend's usual walking time. After hearing three times, you revise your guess toward their average. The stored guess is the model; calculating the average is training; using that number for the next walk is prediction. An average ignores hills and weather, but it is a useful starting point.

Our diagram is: past examples → training procedure → stored rule → new input → prediction. A separate arrow from the true new outcome to the prediction lets us measure error. The new outcome must not flow backward into training before we evaluate it.

## 4. Terminology

A **dataset** is a collection of recorded examples. In a table, a **row** represents one example, such as one day. A **column** records a property, such as temperature. A **feature** is information available when making a prediction. A **target**, or label, is the outcome we want to predict. Tomorrow's eventual sales are a target, not a feature available tonight.

A **model** is a rule mapping inputs to predictions. A **parameter** is a setting estimated during training. **Inference** means applying the trained rule. A **baseline** is a simple reference method that a complicated method should justify improving upon. **Regression** predicts a numerical quantity; **classification** predicts a category. Numerical predictions can include fractional values even when the observed outcome is a count.

## 5. Mathematical foundations

We choose a model that always predicts one number:

$$\hat y=c,\qquad c=\frac{y_1+y_2+\cdots+y_n}{n}.$$

Here $y_i$ is the recorded outcome for example number $i$, in loaves; $n$ is the number of training examples; $c$ is their average, also in loaves; and $\hat y$ means predicted outcome. A hat marks a prediction, not a different unit. The dots mean continue adding all outcomes. Each quantity here is one number, called a scalar.

To measure a mistake, subtract the true outcome from the prediction. Taking its absolute value, written $|\hat y-y|$, removes the sign. A mean absolute error (MAE) averages these nonnegative distances over evaluation examples. Lower is better for this particular objective. It does not distinguish the business costs of overproduction and underproduction.

## 6. Hand-calculated example

Suppose three training days sold 2, 4, and 6 loaves. Add: $2+4+6=12$. Divide by the three days: $c=12/3=4$. Training has now ended. On two new days with sales 5 and 7, the predictions remain 4 and 4. Absolute mistakes are $|4-5|=1$ and $|4-7|=3$. Their mean is $(1+3)/2=2$ loaves.

The average of the two new outcomes is 6. Replacing 4 with 6 before scoring would use information we were supposed to predict. That would be data leakage, even though the arithmetic itself is correct.

## 7. Implementation from scratch

In Python, square brackets collect numbers into a list. The equals sign stores a value under a name; it is an instruction, not an algebraic assertion. `sum` adds numbers, `len` counts them, and `/` divides. Text after `#` explains a line without being executed.

In [ ]:
# Enable embedded figures in this fresh notebook kernel.
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')

In [ ]:
training_sales = [2, 4, 6]
constant = sum(training_sales) / len(training_sales)
test_sales = [5, 7]
errors = [abs(constant - actual) for actual in test_sales]
mae = sum(errors) / len(errors)
print('Learned prediction:', constant, 'loaves')
print('Individual absolute errors:', errors)
print('Test MAE:', mae, 'loaves')
assert constant == 4 and mae == 2

## 8. Step-by-step output inspection

The first output should be 4.0. The decimal point reflects Python's division result, not greater measurement precision. The second output should contain 1.0 and 3.0. `for actual in test_sales` repeats the error calculation for each outcome. `assert` checks a claim and stops execution if it is false. This protects the worked example against accidental changes.

Notice that no repeated training loop was needed. Some algorithms learn with one direct calculation; others improve parameters over many steps. Training is the estimation activity, not necessarily a long computation.

## 9. Library implementation

NumPy stores numerical tables; scikit-learn provides tested prediction tools. `import` makes their functions available. The double brackets below form a table with three rows and one column. DummyRegressor ignores these feature values, but its interface still expects a feature table. `fit` estimates the average; `predict` applies it.

In [ ]:
import numpy as np
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

X_train = np.array([[1], [2], [3]])
baseline = DummyRegressor(strategy='mean').fit(X_train, training_sales)
predictions = baseline.predict(np.array([[4], [5]]))
np.testing.assert_allclose(predictions, [4, 4])
assert mean_absolute_error(test_sales, predictions) == mae
print('Library predictions:', predictions)

The library handles more input formats and validation than our short calculation. It does not make this constant model intelligent about day number.

## 10. Visualization

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

fig, ax = plt.subplots(figsize=(6, 3))
ax.scatter([1, 2, 3], training_sales, label='Training observations')
ax.scatter([4, 5], test_sales, marker='x', label='Held-out observations')
ax.axhline(constant, color='orange', label='Learned constant')
ax.set(xlabel='Day number', ylabel='Sales (loaves)', title='A baseline ignores changing demand')
ax.legend()
display(fig)
plt.close(fig)

The horizontal line shows identical predictions. Vertical distance to an observed point is the prediction mistake. The rising points suggest that ignoring day number may be inadequate; five invented observations cannot establish a dependable trend.

## 11. A realistic question with synthetic data

The next tiny bakery series is invented for reproducibility, not collected from a business. Keeping future days separate imitates forecasting. Real work needs many days, availability information, and checks for holidays and stockouts: sales can understate demand when bread runs out.

In [ ]:
past = np.array([8, 10, 12, 10])
future = np.array([9, 13])
past_mean = past.mean()
future_mae = np.abs(future - past_mean).mean()
print('Past mean:', past_mean, 'Future MAE:', future_mae)
assert past_mean == 10 and future_mae == 2

## 12. Parameters and choices

The learned parameter is the constant 10. Choosing a mean rather than a median is our modeling choice, called a hyperparameter in a configurable estimator. A median is the middle value after sorting; it is less affected by an extreme observation. Mean prediction minimizes squared training error; median prediction minimizes absolute training error. Later mathematics derives this distinction.

## 13. Evaluation

MAE of two loaves means the average magnitude of error on these two days was two. It does not mean every error was two or that tomorrow's error will be two. Compare against business tolerance, another baseline, and more held-out observations. Do not repeatedly choose a model based on the test set; use a separate validation set for choices.

## 14. Assumptions and limitations

This baseline assumes the recent average is relevant to future demand. It cannot adapt predictions to temperature or holidays. Distribution shift means future conditions differ from the conditions represented in training. More complicated models can also fail under such change. Data quality and evaluation design matter before sophistication.

## 15. Common mistakes and debugging

Do not average training and test sales together. Do not confuse a feature with an outcome recorded afterward. Check units before comparing errors: two loaves and two thousand loaves are different. If your assertion fails, print the inputs and intermediate arithmetic before editing the expected answer. An empty training list cannot supply an average.

## 16. Comparison with alternatives

A fixed business rule requires no training and can be preferable when reliable constraints determine the answer. A last-observed-value baseline responds to recent changes. Linear regression can learn how features relate to outcomes but requires more concepts. We will derive it after algebra and derivatives. Machine learning is one tool, not the default answer to every decision.

## 17. Exercises

1. **Beginner:** Name a feature and target for predicting bus travel time, and state when each becomes available.
2. **Beginner:** Train a constant on sales 3, 6, 9. Predict a new day before looking at its outcome.
3. **Beginner:** Compute absolute errors for that prediction when new sales are 4 and 10.
4. **Intermediate:** Replace the final training value with 90. Compare the mean with the median and explain the difference.
5. **Intermediate:** Explain why today's final sales cannot be used tonight to predict today's morning demand.
6. **Challenge:** Design a fair comparison of a mean baseline and a last-day baseline for the next month without using future information.

## 18. Detailed solutions

1. Route length known before departure can be a feature; elapsed arrival time observed afterward is the target. Recorded arrival traffic may be unavailable at departure.
2. The sum is 18 and the count is three, so the prediction is six loaves regardless of the new day's features.
3. The errors are $|6-4|=2$ and $|6-10|=4$; MAE is three loaves.
4. The new mean is $(3+6+90)/3=33$; the median remains six. The mean responds strongly to the extreme day. Neither is automatically correct if 90 signals a real permanent change.
5. The proposed information arrives after the prediction deadline. Its apparent usefulness comes from seeing the answer too early.
6. Pick a historical training window, record both forecasts before each evaluation day's sales are visible, and compare paired errors over the month. Choose any window length using earlier validation periods. Keep the final month separate from those choices, and report stockout and holiday limitations.

## 19. Knowledge check

**What is learned here?** One average, not the sales records themselves. **What is a row?** One day in this example. **Why hold out data?** To inspect performance on observations not used to estimate the rule. **Is zero training error enough?** No; a rule can memorize past outcomes and fail on new days. **Is a prediction a fact?** No; it is a model output with uncertainty.

## 20. Interview preparation

**How would you explain ML without jargon?** We use examples to choose a rule's settings. **Why start with a baseline?** It establishes whether complexity improves something measurable. **Training versus inference?** Training estimates settings; inference reuses them. **What is leakage?** Using unavailable or evaluation information during fitting or choices. **Why can test error change tomorrow?** A finite test sample is uncertain, and future conditions may shift; one score is not a guarantee.

## 21. Summary and next steps

You constructed a model, learned its only parameter, made new predictions, and interpreted mistakes. Continue to [ML types and terminology](02_machine_learning_types_and_terminology.ipynb). Keep the distinction between an input available now and an outcome observed later in mind throughout the academy.